In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import torch.nn.init as init
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import wandb
from enum import Enum


In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/vinayak/.netrc.
wandb: Currently logged in as: cs25m543 (cs25m543-iit-madras) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

### **Data Loader**

In [11]:
from subpolicy import SubPolicy
from cifarpolicy import CIFAR10Policy
from cutout import Cutout

In [12]:
def GetCifar10(batchsize):
    trans_t = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                  transforms.RandomHorizontalFlip(),
                                  # CIFAR10Policy(),
                                  transforms.ToTensor(),
                                  transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                                  Cutout(n_holes=1, length=16)
                                  ])
    trans = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])
    train_data = datasets.CIFAR10('./data', train=True, transform=trans_t, download=False)
    test_data = datasets.CIFAR10('./data', train=False, transform=trans, download=False)
    train_dataloader = DataLoader(train_data, batch_size=batchsize, shuffle=True, num_workers=2)
    test_dataloader = DataLoader(test_data, batch_size=batchsize, shuffle=False, num_workers=2)
    return train_dataloader, test_dataloader

In [13]:
class Activation(Enum):
    RELU = "relu"
    SIGMOID = "sigmoid"
    TANH = "tanh"
    SILU = "silu"
    GELU = "gelu"

    def build(self) -> nn.Module:
        return {
            Activation.RELU: nn.ReLU(inplace=True),
            Activation.SIGMOID: nn.Sigmoid(),
            Activation.TANH: nn.Tanh(),
            Activation.SILU: nn.SiLU(inplace=True),
            Activation.GELU: nn.GELU(),
        }[self]

In [14]:
config = {
        "learning_rate": 1e-3,
        "batch_size": 64,
        "epochs": 5,
        "activation":'sigmoid',
        "optimizer": 'Adam',
        "batch_norm": True,
}


run = wandb.init(
        project="pytorch-starter",
        config=config,
        mode="online",
)

In [15]:
class VGG(nn.Module):
    def __init__(self, num_classes=10,activation_fuction=Activation,batch_norm=False):
        super(VGG, self).__init__()
        self.batch_norm = batch_norm
        self.activation_fuction = activation_fuction

        classifer_layers = [nn.Linear(128, 128)]
        if batch_norm:
            classifer_layers.append(nn.BatchNorm1d(128))
        classifer_layers += [self.activation_fuction.build(), nn.Linear(128, num_classes)]
        self.classifier = nn.Sequential(*classifer_layers)

        self.features = self.make_layers([64,64,'M',128,128,'M'])
        self._initialize_weights()
        
    def make_layers(self,cfg):
        layers = []
        in_channels = 3
        for v in cfg:
            if v == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
                if self.batch_norm:
                    layers += [conv2d, nn.BatchNorm2d(v),self.activation_fuction.build()]
                else:
                    layers += [conv2d, self.activation_fuction.build()]
                in_channels = v
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))  # fixed output size
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity=self.activation_fuction.value)
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()


model = VGG(num_classes=10, activation_fuction=Activation(config["activation"]),batch_norm=config["batch_norm"])
print(model)


VGG(
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): Sigmoid()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): Sigmoid()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): Sigmoid()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (9): Sigmoid()
    (10): Conv2d(128, 128, 

In [16]:
def eval(model,data):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in data:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            _, predicted = outputs.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    acc = 100. * correct / total
    return acc

In [17]:
train_loader,test_loader = GetCifar10(config["batch_size"])

In [18]:
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')

In [19]:
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"])

In [20]:
def train_model(model,epochs,optimizer,train_loader,test_loader,wandb_runner=run,):
   
    for epoch in range(epochs):
      model.train()
      running_loss = 0.0
      for data, target in train_loader:
          data, target = data.to(device), target.to(device)
          optimizer.zero_grad()
          outputs = model(data)
          loss = criterion(outputs, target)
          loss.backward()
          optimizer.step()
          running_loss += loss.item()
      train_acc = eval(model,train_loader)
      test_acc = eval(model,test_loader)
      train_loss = running_loss / len(train_loader)

      wandb_runner.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_acc": test_acc,
        })
      
      print(f"Epoch {epoch} - Train_Loss: {train_loss:.4f} , Train_acc: {train_acc}, Test_acc : {test_acc}")
    run.summary["final_val_acc"] = test_acc

In [ ]:
run.watch(model, log="all", log_freq=100)
train_model(model,config["epochs"],optimizer,train_loader,test_loader,wandb_runner=run)
run.finish()

##Task:

    Change the network architecure, activation function , optimizer, learning rate ,etc. and record the performance of the model.